# 2.8 Dizi Sıralama

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/02-numpy/08-sorting.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: 08 Sorting Arrays

NumPy dizilerindeki değerleri sıralamayla ilgili algoritmalar. Bilgisayar biliminde insertion sort, selection sort, merge sort, quick sort, bubble sort… Hepsi benzer görevi yapar: liste veya dizideki değerleri sıralamak.

Python'da listeler için yerleşik sıralama vardır. sorted sıralı kopya döndürür:


In [ ]:
# python_sorted.py
L = [3, 1, 4, 1, 5, 9, 2, 6]
sorted(L)  # sıralı kopya



Liste metodu sort yerinde sıralar:


In [ ]:
# python_sort_inplace.py
L.sort()  # yerinde; None döner
print(L)



Python sıralaması her türlü yinelemeli nesnede çalışır — örneğin string:


In [ ]:
# sorted_string.py
sorted('python')



Python değerlerinin dinamizmi, uniform sayı dizileri için tasarlanmış NumPy rutinlerinden daha yavaştır — bu yüzden NumPy sıralama rutinleri devreye girer.

Basit selection sort (seçmeli sıralama) listeden tekrar tekrar minimum değeri bulur ve yer değiştirir:


In [ ]:
# selection_sort_fn.py
import numpy as np

def selection_sort(x):
    for i in range(len(x)):
        swap = i + np.argmin(x[i:])
        (x[i], x[swap]) = (x[swap], x[i])
    return x



In [ ]:
# selection_sort_demo.py
x = np.array([2, 1, 4, 3, 5])
selection_sort(x)



Selection sort basitliği için faydalıdır ama büyük diziler için çok yavaştır. $N$ değer için $N$ döngü, her biri $\sim N$ karşılaştırma — ortalama $\mathcal{O}[N^2]$ (Big-O Notasyonu). Eleman sayısını iki katına çıkarırsanız süre yaklaşık dört kat artar.

Selection sort bile favori algoritmam bogosort'tan çok daha iyidir:


In [ ]:
# bogosort_fn.py
def bogosort(x):
    while np.any(x[:-1] > x[1:]):
        np.random.shuffle(x)
    return x



In [ ]:
# bogosort_demo.py
x = np.array([2, 1, 4, 3, 5])
bogosort(x)



Bu algoritma saf şansa dayanır: diziyi rastgele karıştırıp sıralı olana kadar tekrarlar. Ortalama $\mathcal{O}[N 	imes N!]$ — gerçek hesaplamada asla kullanılmamalıdır.

Neyse ki Python ve NumPy'da çok daha verimli yerleşik sıralama vardır.

## NumPy'da Hızlı Sıralama: np.sort ve np.argsort

Python'un sort ve sorted fonksiyonları listeler içindir; NumPy'nin np.sort'u uniform sayı dizileri için çok daha verimlidir. Varsayılan $\mathcal{O}[N\log N]$ quicksort; mergesort ve heapsort da mevcuttur.


In [ ]:
# np_sort.py
x = np.array([2, 1, 4, 3, 5])
np.sort(x)



Yerinde sıralama — dizi metodu sort:


In [ ]:
# sort_inplace.py
x.sort()
print(x)



İlgili fonksiyon argsort — sıralı elemanların indekslerini döndürür:


In [ ]:
# argsort.py
x = np.array([2, 1, 4, 3, 5])
i = np.argsort(x)
print(i)



İlk eleman en küçüğün indeksi, ikinci ikinci en küçüğün… Fancy indexing ile sıralı dizi:


In [ ]:
# argsort_fancy.py
x[i]



Bu bölümün ilerleyen kısmında argsort uygulaması göreceksiniz.

### Satır veya Sütun Boyunca Sıralama


In [ ]:
# sort_X.py
rng = np.random.default_rng(seed=42)
X = rng.integers(0, 10, (4, 6))
print(X)



In [ ]:
# X'in her sütununu sırala
np.sort(X, axis=0)



In [ ]:
# X'in her satırını sırala
np.sort(X, axis=1)



Her satır veya sütun bağımsız dizi gibi ele alınır — satır/sütun değerleri arasındaki ilişki kaybolur!

## Kısmi Sıralama: Partitioning

Bazen tüm diziyi sıralamak değil, k en küçük değeri bulmak yeter. np.partition diziyi alır ve $K$ verir; sonuçta en küçük $K$ değer solda, geri kalan sağda (iç sıralama keyfi):


In [ ]:
# partition.py
x = np.array([7, 2, 3, 1, 6, 5, 4])
np.partition(x, 3)



Çok boyutlu dizide herhangi bir eksen boyunca:


In [ ]:
# partition_axis.py
np.partition(X, 2, axis=1)



np.argsort gibi np.argpartition da partition indekslerini verir.

## Örnek: k-En Yakın Komşular

argsort'u çok eksenli kullanarak her noktanın en yakın komşularını bulalım. 10 rastgele 2B nokta ($10 	imes 2$):


In [ ]:
# knn_X.py
X = rng.random((10, 2))



Her nokta çifti arasındaki kare mesafe. İki nokta arası kare mesafe boyutlar boyunca kare farkların toplamı; broadcasting ve agregasyon ile tek satırda:


In [ ]:
# knn_dist.py
dist_sq = np.sum((X[:, np.newaxis, :] - X[np.newaxis, :, :]) ** 2, axis=-1)



Karmaşık görünüyorsa adım adım:


In [ ]:
# knn_step1.py
differences = X[:, np.newaxis, :] - X[np.newaxis, :]
print("differences.shape:", differences.shape)



In [ ]:
# knn_step2.py
sq_differences = differences ** 2
print("sq_differences.shape:", sq_differences.shape)



In [ ]:
# knn_step3.py
dist_sq = sq_differences.sum(-1)
print("dist_sq.shape:", dist_sq.shape)



Köşegen (her noktanın kendine mesafesi) sıfır olmalı:


In [ ]:
# knn_diag.py
dist_sq.diagonal()



Her satır boyunca argsort — sol sütunlar en yakın komşu indeksleri:


In [ ]:
# knn_nearest.py
nearest = np.argsort(dist_sq, axis=1)
print(nearest)



İlk sütun 0–9 sırası: her noktanın en yakın komşusu kendisi.

Sadece en yakın $k$ komşu gerekiyorsa tam sıralama fazla — np.argpartition:


In [ ]:
# knn_partition.py
K = 2
nearest_partition = np.argpartition(dist_sq, K + 1, axis=1)



Her noktadan iki en yakın komşuya çizgi çizilebilir (Matplotlib). Bazı noktalardan iki satırdan fazla çizgi çıkabilir: A, B'nin en yakın komşusu olsa B, A'nın en yakın komşusu olmak zorunda değildir.

Döngü yazmak cazip gelebilir ama vektörize sürüm çok daha verimlidir; girdi boyutundan bağımsız aynı kod 100 veya 1.000.000 noktada çalışır. Çok büyük aramalarda KD-Tree gibi $\mathcal{O}[N\log N]$ algoritmalar vardır — Scikit-Learn KDTree.

## Ayrıntı: Big-O Notasyonu

Big-O notasyonu bir algoritmanın girdi büyüdükçe işlem sayısının nasıl ölçeklendiğini tanımlar. Teoride small-o, big-$	heta$, big-$\Omega$ ayrımları vardır; pratikte veri biliminde genelde daha gevşek yorum kullanılır: algoritmanın ölçeklenmesinin genel tanımı.

$\mathcal{O}[N]$ algoritma $N=1000$ için 1 saniye sürüyorsa $N=5000$ için kabaca 5 saniye beklenir. $\mathcal{O}[N^2]$ algoritma $N=1000$ için 1 saniye ise $N=5000$ için yaklaşık 25 saniye.

Big-O tek başına duvar saati süresini söylemez — yalnızca $N$ değişince ölçeklenmeyi. Küçük veride $\mathcal{O}[N^2]$ algoritma 0,01 s, $\mathcal{O}[N]$ algoritma 1 s sürebilir; $N$ 1000 kat artınca $\mathcal{O}[N]$ kazanır.

Milyarlarca örnekte $\mathcal{O}[N]$ ile $\mathcal{O}[N^2]$ farkı kritiktir. Kitap boyunca algoritma karşılaştırmalarında bu notasyonu kullanacağız.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      np.argpartition ile dizideki 3 en küçük elemanın indekslerini bulun.
    
      import numpy as np
x = np.array([7, 2, 9, 1, 5, 3])
i = np.argpartition(x, 3)[:3]
print("3 en küçük:", x[i], "  indeksler:", i)

> **Not**
>
